In [ ]:
import pandas as pd

PATH_TO_DATA="/lustre/fs5/jarv_lab/scratch/adenisova/Inno_2025/conserved_regions/"
PATH_TO_WGS="/lustre/fs5/jarv_lab/scratch/adenisova/Inno_2025/genome_data/vgpBirds_137_way"

df = pd.read_csv(
    "/lustre/fs5/jarv_lab/scratch/adenisova/Inno_2025/initial_data/ALLBIRDTRAITS_Database_March_2026.csv"
)

In [ ]:
cons_all = pd.read_csv(
    f"{PATH_TO_DATA}/Sackton_et_al/selected_elements/final_cnees_long.bed",
    sep="\t",
    names = ["chr", "start", "end", "cnee_id"]
)

cons_ATAC = pd.read_csv(
    f"{PATH_TO_DATA}/Sackton_et_al/selected_elements/cnee_overlaps_ATAC_Seq.bed",
    sep="\t",
    names = ["chr", "start", "end", "cnee_id"]
)
print(len(set(cons_all["cnee_id"]) & set(cons_ATAC["cnee_id"])))

print(len(set(cons_ATAC["cnee_id"])))

60665
60665


In [4]:
cons_all

,chr,start,end,cnee_id
0,NC_006088.3,128730,128810,mCE274
1,NC_006088.3,129142,129216,mCE275
2,NC_006088.3,129731,129802,mCE276
3,NC_006088.3,150042,150104,mCE296
4,NC_006088.3,163335,163398,mCE305
...,...,...,...,...
283996,NW_003764414.1,523,575,mCE1956209
283997,NW_003764451.1,18269,18387,mCE1956230
283998,NW_003764451.1,18397,18532,mCE1956231
283999,NW_003764451.1,18807,19146,mCE1956234


In [24]:
print("mCE445" in set(cons_ATAC["cnee_id"]))

cons_all[cons_all["cnee_id"] == "mCE445"]

False


,chr,start,end,cnee_id
13,NC_006088.3,408467,408632,mCE445


In [25]:
cons_all["chr"].unique()

array(['NC_006088.3', 'NC_006089.3', 'NC_006090.3', 'NC_006091.3',
       'NC_006092.3', 'NC_006093.3', 'NC_006094.3', 'NC_006095.3',
       'NC_006096.3', 'NC_006097.3', 'NC_006098.3', 'NC_006099.3',
       'NC_006100.3', 'NC_006101.3', 'NC_006102.3', 'NC_006103.3',
       'NC_006104.3', 'NC_006105.3', 'NC_006106.3', 'NC_006107.3',
       'NC_006108.3', 'NC_006109.3', 'NC_006110.3', 'NC_006111.3',
       'NC_006112.2', 'NC_006113.3', 'NC_006114.3', 'NC_006115.3',
       'NC_006126.3', 'NC_006127.3', 'NC_008465.2', 'NC_008466.2',
       'NW_003764410.1', 'NW_003764413.1', 'NW_003764414.1',
       'NW_003764451.1', 'NW_003764493.1'], dtype=object)

In [3]:
df_sp_from_alignmnet = pd.read_csv(
    "/lustre/fs5/jarv_lab/scratch/adenisova/Inno_2025/initial_data/TreeClusterMax0.3.tsv",
    sep = "\t"
)
df_birds_alignment = df_sp_from_alignmnet[df_sp_from_alignmnet["Lineage"] == 'Birds']


In [8]:
from ete3 import Tree

a = Tree(
    f"{PATH_TO_WGS}/roadies_v1.1.16b.nwk"
)
leaf_names = set(a.get_leaf_names())
a.prune(leaf_names.intersection(set(df_birds_alignment["# accession"])))

leaf_names_birds = set(a.get_leaf_names())
# print(a)

a.write(format=1, outfile=f"{PATH_TO_WGS}/roadies_v1.1.16b_birds_only.nwk")



In [9]:
df["ScientificName"] = df['Species'].apply(lambda x: x.replace("_", " "))
df_merged = df_birds_alignment.merge(df, on="ScientificName", how="left")

"TOTALINNOVATIONS2025_ResEff", "TECHINNO2025_ResEff", "Relative_brain_size", "Wing_length"
df_no_na = df_merged[
    (~df_merged["TECHINNO2025_ResEff"].isna()) & 
    (~df_merged["FOODINNO2025_ResEff"].isna()) &
    (~df_merged["TOTALINNOVATIONS2025_ResEff"].isna()) &
    (~df_merged["Relative_brain_size"].isna()) &
    (~df_merged["Wing_length"].isna())
]

a.prune(leaf_names.intersection(set(df_no_na["# accession"])))
print(a)

# find rer_trees_birds_by_final_cnees_long/ -type f -size -100c -exec rm {} \;


                           /-GCA_016904835.1
                          |
                          |                        /-GCA_047830755.1
                          |                       |
                          |                     /-|      /-GCF_028018845.1
                          |                    |  |   /-|
                          |                    |   \-|   \-GCF_035770615.1
                          |                  /-|     |
                          |                 |  |      \-GCA_031885435.1
                          |                 |  |
                          |                 |  |   /-GCA_009764595.1
                          |               /-|   \-|
                          |              |  |      \-GCF_020745825.1
                          |              |  |
                          |            /-|  |   /-GCF_027477595.1
                          |           |  |   \-|
                          |           |  |      \-GCA_963513975.1
    

In [10]:
for ln in a.iter_leaves():
    ln.name = ln.name.split(".")[0]

a.write(format=1, outfile=f"{PATH_TO_WGS}/roadies_v1.1.16b_birds_only_renamed_no_na.nwk")

In [ ]:
df_merged_for_rer = df_merged[[
    '# accession',
    'ScientificName', 'TOTALINNOVATIONS2025_ResEff', 'FOODINNO2025_ResEff', 
    'TECHINNO2025_ResEff', "Relative_brain_size", "Wing_length"
]]
df_merged_for_rer.rename(columns={"# accession": "sci_name_2025"}, inplace=True)
df_merged_for_rer["sci_name_2025"] = df_merged_for_rer["sci_name_2025"].apply(lambda x: x.split(".")[0])
df_merged_for_rer.to_csv(f"{PATH_TO_DATA}/df_merged_for_rer.csv", index=False)

/tmp/ipykernel_1419559/1937698504.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merged_for_rer.rename(columns={"# accession": "sci_name_2025"}, inplace=True)
/tmp/ipykernel_1419559/1937698504.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merged_for_rer["sci_name_2025"] = df_merged_for_rer["sci_name_2025"].apply(lambda x: x.split(".")[0])


In [ ]:
df_merged_for_rer = df_no_na[[
    '# accession',
    'ScientificName', 'TOTALINNOVATIONS2025_ResEff', 'FOODINNO2025_ResEff', 
    'TECHINNO2025_ResEff', "Relative_brain_size", "Wing_length"
]]
df_merged_for_rer.rename(columns={"# accession": "sci_name_2025"}, inplace=True)
df_merged_for_rer["sci_name_2025"] = df_merged_for_rer["sci_name_2025"].apply(lambda x: x.split(".")[0])
df_merged_for_rer.to_csv(f"{PATH_TO_DATA}/df_merged_for_rer_no_na.csv", index=False)

In [24]:
df_sp_from_alignmnet[df_sp_from_alignmnet["# accession"].apply(lambda x: x in leaf_names_birds)]

,# accession,Cluster,Lineage,Superorder,Orders,Family,ScientificName,EnglishName
152,GCF_036417845.1,20,Birds,Palaeognathae,Apterygiformes,Apterygidae,Apteryx mantelli,North Island brown kiwi
153,GCF_036370855.1,20,Birds,Palaeognathae,Casuariiformes,Dromaiidae,Dromaius novaehollandiae,emu
154,GCF_028389875.1,20,Birds,Palaeognathae,Rheiformes,Rheidae,Rhea pennata,lesser rhea
155,GCF_040807025.1,20,Birds,Palaeognathae,Struthioniformes,Struthionidae,Struthio camelus australis,common ostrich
156,GCA_047922985.1,20,Birds,Palaeognathae,Tinamiformes,Tinamidae,Eudromia elegans,elegant crested tinamou
...,...,...,...,...,...,...,...,...
459,GCA_009819595.1,39,Birds,Core landbirds,Coraciiformes,Meropidae,Merops nubicus,carmine bee-eater
460,GCA_034619465.1,39,Birds,Core landbirds,Leptosomiiformes,Leptosomidae,Leptosomus discolor,cuckoo-roller
461,GCF_015220805.1,39,Birds,Core landbirds,Piciformes,Lybiidae,Pogoniulus pusillus,red-fronted tinkerbird
462,GCF_014839835.1,39,Birds,Core landbirds,Piciformes,Picidae,Dryobates pubescens,downy woodpecker


In [9]:
leaf_names

['GCF_028018845.1',
 'GCF_035770615.1',
 'GCA_031885435.1',
 'GCF_027887145.1',
 'GCF_027579445.1',
 'GCA_036010785.1',
 'GCA_047830755.1',
 'GCA_009764595.1',
 'GCF_020745825.1',
 'GCF_027477595.1',
 'GCA_963513975.1',
 'GCF_036417665.1',
 'GCA_048771995.1',
 'GCA_046129695.1',
 'GCF_026979565.1',
 'GCF_029582105.1',
 'GCA_903797595.2',
 'GCF_009819885.2',
 'GCF_963662255.1',
 'GCA_027559615.1',
 'GCA_014839755.1',
 'GCF_009819655.1',
 'GCA_965231275.1',
 'GCF_015227805.1',
 'GCF_030490865.1',
 'GCF_020740725.1',
 'GCF_009650955.1',
 'GCA_965178545.1',
 'GCA_046129655.1',
 'GCA_014706295.1',
 'GCF_009829145.1',
 'GCF_036250125.1',
 'GCA_045364795.1',
 'GCA_016904835.1',
 'GCF_012275295.1',
 'GCF_037157495.1',
 'GCA_963264785.1',
 'GCA_028858755.1',
 'GCA_045345405.1',
 'GCA_039720435.1',
 'GCF_004027225.2',
 'GCF_015220075.1',
 'GCF_023634085.1',
 'GCF_023638135.1',
 'GCF_023634155.1',
 'GCF_017639655.2',
 'GCA_963210335.1',
 'GCA_009819825.1',
 'GCF_028858725.1',
 'GCF_014839835.1',
